# Исторический исследовательский код

Эта версия не является основным протоколом статьи. Выводы удалены. Она может требовать исходные локальные файлы и иную схему данных, которые не включены в открытый набор. Для воспроизведения статьи используйте `notebooks/vascular`. Старые метрики не смешиваются с актуальными.


In [ ]:
from datetime import datetime, date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu, fisher_exact, beta
import statsmodels.api as sm

In [ ]:
df = pd.read_csv("data/обзвон_данные.csv", sep=';')

In [ ]:
display(df)

In [ ]:
df['ФИО'].unique()

## Очистка

In [ ]:
df = df.dropna(how="all").copy()
fio = df["ФИО"].astype("string").str.strip()
df = df.loc[fio.notna() & fio.ne("")].copy()
df = df.iloc[:, :-1].copy()
empty_columns = df.columns[df.apply(lambda col: col.astype("string").str.strip().replace("", pd.NA).isna().all())]
df = df.drop(columns=empty_columns)
print(f"После очистки: {len(df)} строк, {df.shape[1]} столбцов")

## Летальный исход

In [ ]:
outcome_text = df["летальный исход"].astype("string").str.strip()
outcome_numeric = pd.to_numeric(outcome_text, errors="coerce")
outcome_date = pd.to_datetime(outcome_text.where(outcome_numeric.isna()), errors="coerce", dayfirst=True, format="mixed")
alive = outcome_numeric.eq(0).fillna(False)
dead = (outcome_numeric.eq(1) | outcome_numeric.between(1900, 2100) | outcome_date.notna()).fillna(False)
unknown_values = outcome_text.notna() & ~alive & ~dead
if unknown_values.any():
    raise ValueError(f"Неизвестные записи исхода: {outcome_text.loc[unknown_values].unique().tolist()}")
df["летальный исход"] = pd.Series(pd.NA, index=df.index, dtype="Int64")
df.loc[alive, "летальный исход"] = 0
df.loc[dead, "летальный исход"] = 1
print(df["летальный исход"].value_counts(dropna=False).sort_index())

## Сосудистые бассейны

In [ ]:
names = df["ФИО"].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)

def anchor_position(name):
    positions = np.flatnonzero(names.eq(name).fillna(False).to_numpy())
    if len(positions) != 1:
        raise ValueError(f"Имя не найдено однозначно: {name}")
    return positions[0]

boundaries = [
    ("ANONYMIZED_PERSON", "ANONYMIZED_PERSON"),
    ("ANONYMIZED_PERSON", "ANONYMIZED_PERSON"),
    ("ANONYMIZED_PERSON", "ANONYMIZED_PERSON"),
]
positions = [(anchor_position(start), anchor_position(end)) for start, end in boundaries]
if not (positions[0][0] <= positions[0][1] < positions[1][0] <= positions[1][1] < positions[2][0] <= positions[2][1]):
    raise ValueError("Границы сосудистых бассейнов расположены неожиданно")

df_peripheral = df.iloc[positions[0][0]:positions[0][1] + 1].reset_index(drop=True)
df_aortic = df.iloc[positions[1][0]:positions[1][1] + 1].reset_index(drop=True)
df_carotid = df.iloc[positions[2][0]:positions[2][1] + 1].reset_index(drop=True)
df_all = pd.concat([df_peripheral, df_aortic, df_carotid], ignore_index=True)
df = df_all
print(f"Общая: {len(df_all)}, периферия: {len(df_peripheral)}, АБШ/аневризма аорты: {len(df_aortic)}, сонная: {len(df_carotid)}")

## Распределение смертей

In [ ]:
groups = {
    "Периферия": df_peripheral,
    "АБШ/аневризма аорты": df_aortic,
    "Сонная": df_carotid,
    "Общая": df_all,
}
summary = pd.DataFrame([
    {
        "Бассейн": name,
        "Всего": len(part),
        "Живы": int(part["летальный исход"].eq(0).sum()),
        "Умерли": int(part["летальный исход"].eq(1).sum()),
        "Исход неизвестен": int(part["летальный исход"].isna().sum()),
    }
    for name, part in groups.items()
])
summary["Смертность среди известных исходов, %"] = (
    100 * summary["Умерли"] / (summary["Живы"] + summary["Умерли"])
).round(1)
display(summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = ["#527A95", "#B86B7D", "#6F9A87", "#59636E"]
labels = summary["Бассейн"]

axes[0].bar(labels, summary["Умерли"], color=colors)
axes[0].set_title("Число смертей")
axes[0].set_ylabel("Пациенты, n")

for i, value in enumerate(summary["Умерли"]):
    axes[0].text(i, value + 0.5, str(value), ha="center")

rates = summary["Смертность среди известных исходов, %"]
axes[1].bar(labels, rates, color=colors)
axes[1].set_title("Смертность среди пациентов с известным исходом")
axes[1].set_ylabel("%")
axes[1].set_ylim(0, max(rates) * 1.25)

for i, rate in enumerate(rates):
    known = summary.loc[i, "Живы"] + summary.loc[i, "Умерли"]
    axes[1].text(i, rate + 0.4, f"{rate:.1f}%\n(n={known})", ha="center")

for ax in axes:
    ax.tick_params(axis="x", rotation=20)
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

## Общий неблагоприятный исход

`adverse_outcome`: 1 — смерть или хотя бы одно из шести осложнений; 0 — все семь событий явно отсутствуют; пропуск — нет положительных событий, но хотя бы один исход неизвестен. Даты, годы и описания выполненных вмешательств обозначают событие. Исходные столбцы анкеты сохранены; бинарные события создаются в отдельной аналитической таблице.

In [ ]:
import re
from scipy.stats import chi2_contingency, MonteCarloMethod, PermutationMethod
from scipy.stats.contingency import odds_ratio
from statsmodels.stats.multitest import multipletests
from IPython.display import Markdown

event_labels = {
    "death": "Летальный исход", "acs": "ОКС", "stroke": "Инсульт",
    "tia": "ТИА", "amputation": "Ампутация", "rethrombosis": "Ретромбоз",
    "reoperation": "Реоперация",
}
event_sources = ["летальный исход", "ОКС", "Инсульт", "ТИА", "ампутация", "ретромбоз", "реоперация"]

def event_binary(series):
    text = series.astype("string").str.strip().str.lower()
    number = pd.to_numeric(text.str.replace(",", ".", regex=False), errors="coerce")
    result = pd.Series(pd.NA, index=series.index, dtype="Int64")
    result.loc[number.eq(0) | text.isin(["нет", "no"])] = 0
    dated = text.str.contains(r"\b(?:19|20)\d{2}\b|\d{1,2}[./,]\d{1,2}[./,]\d{2,4}", na=False)
    result.loc[number.gt(0) | text.isin(["да", "yes", "+"]) | dated] = 1
    return result

analysis = df_all.copy()
analysis["vascular_group"] = np.repeat(
    ["Периферия", "АБШ/аневризма аорты", "Сонная"],
    [len(df_peripheral), len(df_aortic), len(df_carotid)],
)
for key, source in zip(event_labels, event_sources):
    analysis[key] = event_binary(analysis[source])

events = analysis[list(event_labels)]
analysis["adverse_outcome"] = pd.Series(pd.NA, index=analysis.index, dtype="Int64")
analysis.loc[events.eq(0).fillna(False).all(axis=1), "adverse_outcome"] = 0
analysis.loc[events.eq(1).fillna(False).any(axis=1), "adverse_outcome"] = 1
df_all["adverse_outcome"] = analysis["adverse_outcome"].array
for part, name in zip([df_peripheral, df_aortic, df_carotid], list(groups)[:3]):
    part["adverse_outcome"] = analysis.loc[analysis["vascular_group"].eq(name), "adverse_outcome"].array
df = df_all

## Характеристики пациентов

Исходные факторы присоединяются из `data/База_док_13_08_2026_Коррекция_по_нозологии_финал_.csv` по ФИО и дате операции. При несовпадении написания ФИО используется только однозначное совпадение даты рождения и операции. Противоречивые значения в повторных исходных записях остаются неизвестными. Возраст — полные годы на дату операции; пол определяется по ФИО и требует подтверждения. Не используются синтетические данные, импутация и показатели, возникшие после операции.

СД, ИБС и ФП не проверяются как бинарные факторы: в исходнике преимущественно отмечено только наличие, а пустые ячейки нельзя считать отсутствием. СКФ рассматривается как числовой показатель, а не подтверждённый диагноз ХБП.

Числа с десятичной запятой переведены в числовой формат. Ячейки, ошибочно сохранённые Excel как даты 1899–1900 годов, возвращены в серийные числа Excel; исходный CSV не изменяется. `female`: 1 — женщина, 0 — мужчина; признаки анамнеза: 1 — наличие, 0 — явно указанное отсутствие. Категории ФВ: <40%, 40–50% включительно, >50%. Лабораторные единицы, кроме явно указанных в источнике, требуют подтверждения.

In [ ]:
baseline = pd.read_csv("data/База_док_13_08_2026_Коррекция_по_нозологии_финал_.csv", sep=";")

def parse_dates(series):
    text = series.astype("string").str.strip().str.replace(r"^0(\d{2})-", r"20\1-", regex=True)
    iso = text.str.match(r"^\d{4}-", na=False)
    result = pd.to_datetime(text.where(~iso), format="mixed", dayfirst=True, errors="coerce")
    result.loc[iso] = pd.to_datetime(text.loc[iso], format="ISO8601", errors="coerce")
    return result

def name_key(series):
    return series.astype("string").str.lower().str.replace("ё", "е").str.replace(r"\s+", " ", regex=True).str.strip()

def numeric(series):
    text = series.astype("string").str.replace(",", ".", regex=False)
    date_mask = text.str.match(r"^(?:1899|1900)-\d{2}-\d{2}", na=False)
    parsed = pd.to_datetime(text.where(date_mask), format="ISO8601", errors="coerce")
    serial = (parsed - pd.Timestamp("1899-12-30")).dt.total_seconds() / 86400
    serial = serial.where(serial >= 61, serial - 1)
    result = pd.to_numeric(text.str.extract(r"([-+]?\d+(?:\.\d+)?)", expand=False), errors="coerce")
    return result.where(~date_mask, serial)

def binary(series):
    return series.astype("string").str.strip().str.lower().map(
        {"да": 1, "+": 1, "1": 1, "1.0": 1, "нет": 0, "-": 0, "0": 0, "0.0": 0}
    ).astype("Int64")

def sex_from_name(value):
    if pd.isna(value):
        return np.nan
    tokens = re.findall(r"[а-яё]+", str(value).lower())
    if any(t.endswith(("вна", "ична")) for t in tokens):
        return 1
    if any(t.endswith(("вич", "ьич")) for t in tokens):
        return 0
    if tokens and tokens[0].endswith(("ова", "ева", "ина", "ая", "ская")):
        return 1
    if tokens and tokens[0].endswith(("ов", "ев", "ин", "ий", "ый", "ской")):
        return 0
    return np.nan

feature_sources = {
    "prior_vascular_surgery": "любая операция на сосудах до индексного события",
    "mi_history": "Инфаркт анамнез (да/нет)", "stroke_history": "ОНМК анамнез (да/нет)",
    "ef_percent": "Узи сердца ФВ %", "creatinine_value": "креатинин ",
    "total_cholesterol": "холестерин", "ldl_mmol_l": "ЛПНП, ммоль/л",
    "glucose_value": "Глюкоза", "crp_value": "СРБ", "egfr_value": "СКФ", "bmi": "ИМТ",
}
binary_features = ["female", "prior_vascular_surgery", "mi_history", "stroke_history"]
continuous_features = ["age_years", "ef_percent", "creatinine_value", "total_cholesterol",
                       "ldl_mmol_l", "glucose_value", "crp_value", "egfr_value", "bmi"]
base_features = pd.DataFrame(index=baseline.index)
base_features["_name"] = name_key(baseline["ФИО"])
base_features["_operation"] = parse_dates(baseline["Дата операции"])
base_features["_birth"] = parse_dates(baseline["Год рождения"])
for key, source in feature_sources.items():
    base_features[key] = binary(baseline[source]) if key in binary_features else numeric(baseline[source])
base_features["ef_percent"] = base_features["ef_percent"].where(
    base_features["ef_percent"] > 1.5, base_features["ef_percent"] * 100
)
base_features["ef_percent"] = base_features["ef_percent"].where(base_features["ef_percent"].between(10, 90))

def unique_value(series):
    values = series.dropna().unique()
    return values[0] if len(values) == 1 else np.nan

analysis["_name"] = name_key(analysis["ФИО"])
analysis["_operation"] = parse_dates(analysis["Дата операции"])
analysis["_birth"] = parse_dates(analysis["Год рождения"])
source_columns = list(feature_sources)
by_name = base_features.dropna(subset=["_name", "_operation"]).groupby(
    ["_name", "_operation"]
)[source_columns].agg(unique_value)
by_dates = base_features.dropna(subset=["_birth", "_operation"]).groupby(
    ["_birth", "_operation"]
).filter(lambda part: part["_name"].nunique() == 1).groupby(
    ["_birth", "_operation"]
)[source_columns].agg(unique_value)
matched = analysis.join(by_name, on=["_name", "_operation"])
fallback = analysis.join(by_dates, on=["_birth", "_operation"])
name_match = pd.MultiIndex.from_frame(analysis[["_name", "_operation"]]).isin(by_name.index)
for key in source_columns:
    analysis[key] = matched[key].where(name_match, fallback[key])

operation = analysis["_operation"]
birth = analysis["_birth"]
birthday_later = (operation.dt.month < birth.dt.month) | (
    operation.dt.month.eq(birth.dt.month) & (operation.dt.day < birth.dt.day)
)
age = operation.dt.year - birth.dt.year - birthday_later.astype(int)
analysis["age_years"] = age.where(age.between(18, 105)).astype("Int64")
analysis["female"] = analysis["ФИО"].map(sex_from_name).astype("Int64")
analysis["ef_category"] = pd.cut(analysis["ef_percent"], [-np.inf, 40, 50, np.inf],
                                labels=["<40", "40–50", ">50"], right=False).astype("string")
analysis.loc[analysis["ef_percent"].eq(50), "ef_category"] = "40–50"
analysis = analysis.drop(columns=["_name", "_operation", "_birth"])
analysis_groups = {name: analysis.loc[analysis["vascular_group"].eq(name)].copy() for name in list(groups)[:3]}
analysis_groups["Общая"] = analysis
feature_labels = {
    "age_years": "Возраст, лет", "female": "Женский пол",
    "prior_vascular_surgery": "Операция на сосудах в анамнезе",
    "mi_history": "ИМ в анамнезе", "stroke_history": "ОНМК в анамнезе",
    "ef_percent": "ФВ, %", "ef_category": "Категория ФВ",
    "creatinine_value": "Креатинин", "total_cholesterol": "Общий холестерин",
    "ldl_mmol_l": "ЛПНП, ммоль/л", "glucose_value": "Глюкоза",
    "crp_value": "С-реактивный белок", "egfr_value": "СКФ", "bmi": "ИМТ",
}
display(pd.DataFrame({"Переменная": list(feature_labels), "Значение": list(feature_labels.values())}))

## Частота общего исхода

In [ ]:
def frequency(part, outcome):
    yes = int(part[outcome].eq(1).sum())
    no = int(part[outcome].eq(0).sum())
    return {"Всего": len(part), "Событие": yes, "Нет события": no,
            "Неизвестно": int(part[outcome].isna().sum()),
            "Среди известных, %": 100 * yes / (yes + no) if yes + no else np.nan}

adverse_summary = pd.DataFrame([
    {"Бассейн": name, **frequency(part, "adverse_outcome")}
    for name, part in analysis_groups.items()
])
display(adverse_summary.round(1))
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(adverse_summary["Бассейн"], adverse_summary["Среди известных, %"], color=colors)
for bar, row in zip(bars, adverse_summary.to_dict("records")):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{row["Среди известных, %"]:.1f}%\n{row["Событие"]}/{row["Событие"] + row["Нет события"]}', ha="center")
ax.set(ylabel="Среди известных исходов, %", title="Смерть или хотя бы одно осложнение")
ax.set_ylim(0, adverse_summary["Среди известных, %"].max() + 12)
ax.tick_params(axis="x", labelrotation=12)
plt.tight_layout()
plt.show()

## Статистические сравнения

Числовые показатели: медиана [Q1; Q3], двусторонний критерий Манна–Уитни; при малых группах — перестановочный вариант. `r_rb = 2U/(n₁n₀) − 1`: положительное значение означает большие значения показателя в группе события. Бинарные показатели: χ² при всех ожидаемых частотах ≥5, иначе Фишер; условное отношение шансов (ОШ) и 95% ДИ. Для категорий ФВ при малых частотах используется обобщённый тест Фишера с Монте-Карло, 9999 повторений, фиксированное начальное состояние.

`p_value` — исходное p; `q_value` — поправка Бенджамини–Хохберга: отдельно все сравнения факторов с общим исходом, отдельно все сравнения факторов с индивидуальными событиями. q<0,05 выделено как более устойчивый результат. Пропуски исключаются только из конкретного сравнения; n₀/n₁ относятся к доступным данным.

Методы: [SciPy](https://docs.scipy.org/doc/scipy/reference/stats.html), [поправка FDR](https://www.statsmodels.org/stable/generated/statsmodels.stats.multitest.multipletests.html).

In [ ]:
def median_iqr(values):
    if len(values) == 0:
        return "—"
    q1, med, q3 = np.quantile(values, [0.25, 0.5, 0.75])
    return f"{med:.2f} [{q1:.2f}; {q3:.2f}]"

def association_table(part, outcome):
    rows = []
    for feature in continuous_features + binary_features + ["ef_category"]:
        data = part[[feature, outcome]].dropna()
        y = data[outcome].astype(int)
        x = data[feature]
        n0, n1 = int(y.eq(0).sum()), int(y.eq(1).sum())
        row = {"Фактор": feature_labels[feature], "n₀": n0, "n₁": n1,
               "Без события": "—", "С событием": "—", "Эффект": "—",
               "Тест": "Недостаточно вариации", "p_value": np.nan}
        if feature in continuous_features:
            a, b = x.loc[y.eq(1)].to_numpy(float), x.loc[y.eq(0)].to_numpy(float)
            row.update({"Без события": median_iqr(b), "С событием": median_iqr(a)})
            if min(n0, n1) >= 2 and x.nunique() > 1:
                method = PermutationMethod(n_resamples=9999, rng=np.random.default_rng(42)) if min(n0, n1) < 10 else "asymptotic"
                test = mannwhitneyu(a, b, alternative="two-sided", method=method)
                row.update({"Эффект": f"r_rb={2*test.statistic/(n1*n0)-1:.3f}",
                            "Тест": "Манна–Уитни" if isinstance(method, str) else "Манна–Уитни, перестановки",
                            "p_value": test.pvalue})
        elif feature in binary_features:
            a, b = int((x.eq(1) & y.eq(1)).sum()), int((x.eq(1) & y.eq(0)).sum())
            row.update({"Без события": f"{b}/{n0}" if n0 else "—", "С событием": f"{a}/{n1}" if n1 else "—"})
            if min(n0, n1) > 0 and x.nunique() > 1:
                table = np.array([[a, b], [n1-a, n0-b]])
                chi = chi2_contingency(table, correction=False)
                p = chi.pvalue if (chi.expected_freq >= 5).all() else fisher_exact(table).pvalue
                estimate = odds_ratio(table)
                ci = estimate.confidence_interval()
                row.update({"Эффект": f"ОШ={estimate.statistic:.2f} [{ci.low:.2f}; {ci.high:.2f}]",
                            "Тест": "χ²" if (chi.expected_freq >= 5).all() else "Фишер", "p_value": p})
        elif min(n0, n1) > 0 and x.nunique() > 1:
            table = pd.crosstab(x, y).reindex(columns=[0, 1], fill_value=0)
            chi = chi2_contingency(table.to_numpy(), correction=False)
            if (chi.expected_freq >= 5).all():
                p, method = chi.pvalue, "χ²"
            else:
                p = fisher_exact(table.to_numpy(), method=MonteCarloMethod(
                    n_resamples=9999, rng=np.random.default_rng(42))).pvalue
                method = "Фишер, Монте-Карло"
            row.update({"Без события": "; ".join(f"{k}: {v}" for k, v in table[0].items()),
                        "С событием": "; ".join(f"{k}: {v}" for k, v in table[1].items()),
                        "Тест": method, "p_value": p})
        rows.append(row)
    return pd.DataFrame(rows)

def add_fdr(table):
    table = table.copy()
    valid = table["p_value"].notna()
    table["q_value"] = np.nan
    if valid.any():
        table.loc[valid, "q_value"] = multipletests(table.loc[valid, "p_value"], method="fdr_bh")[1]
    return table

def show_tests(table):
    display(table.style.format({"p_value": "{:.4f}", "q_value": "{:.4f}"}, na_rep="—").map(
        lambda value: "background-color: #dceede" if pd.notna(value) and value < 0.05 else "",
        subset=["q_value"],
    ))

## Факторы общего исхода

In [ ]:
adverse_associations = add_fdr(pd.concat([
    association_table(part, "adverse_outcome").assign(Бассейн=name)
    for name, part in analysis_groups.items()
], ignore_index=True))
for name in analysis_groups:
    display(Markdown(f"### {name}"))
    show_tests(adverse_associations.loc[adverse_associations["Бассейн"].eq(name)]
               .drop(columns="Бассейн").sort_values("p_value"))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, feature in zip(axes, ["age_years", "ldl_mmol_l", "ef_percent"]):
    plot = analysis.dropna(subset=[feature, "adverse_outcome"]).copy()
    plot["Исход"] = plot["adverse_outcome"].map({0: "Нет событий", 1: "Есть событие"})
    sns.boxplot(data=plot, x="vascular_group", y=feature, hue="Исход", ax=ax,
                palette={"Нет событий": "#527A95", "Есть событие": "#B86B7D"})
    ax.set(xlabel="", ylabel=feature_labels[feature])
    ax.tick_params(axis="x", labelrotation=15)
    ax.legend(title="", fontsize=8)
fig.suptitle("Характеристики по бассейнам и общему исходу")
plt.tight_layout()
plt.show()

## Отдельные события

Одно наблюдение может иметь несколько событий: их числа не складываются в число неблагоприятных исходов. Процент каждого события рассчитывается только среди записей с известным значением именно этого события.

In [ ]:
event_summary = pd.DataFrame([
    {"Бассейн": name, "Событие": event_labels[key],
     "Число событий": frequency(part, key)["Событие"],
     "Известно": int(part[key].notna().sum()),
     "Неизвестно": int(part[key].isna().sum()),
     "Частота, %": frequency(part, key)["Среди известных, %"]}
    for name, part in analysis_groups.items() for key in event_labels
])
display(event_summary.round(1))
matrix = event_summary.pivot(index="Событие", columns="Бассейн", values="Частота, %").reindex(
    index=list(event_labels.values()), columns=list(analysis_groups)
)
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(matrix, annot=True, fmt=".1f", cmap="Reds", linewidths=0.5,
            cbar_kws={"label": "Среди известных, %"}, ax=ax)
ax.set(title="Смерть и осложнения по сосудистым бассейнам", xlabel="", ylabel="")
plt.tight_layout()
plt.show()

## Связи факторов с отдельными событиями

Таблицы p отображают все проверяемые связи. Ниже отдельно выведены эффекты с исходным p<0,05; они не считаются устойчивыми, если q≥0,05. Полная таблица, включая размеры сравниваемых групп и эффекты, доступна в `event_associations`.

In [ ]:
event_associations = add_fdr(pd.concat([
    association_table(part, key).assign(Бассейн=name, Событие=label)
    for name, part in analysis_groups.items() for key, label in event_labels.items()
], ignore_index=True))
for name in analysis_groups:
    display(Markdown(f"### {name}: p_value"))
    table = event_associations.loc[event_associations["Бассейн"].eq(name)]
    display(table.pivot(index="Фактор", columns="Событие", values="p_value").reindex(
        columns=list(event_labels.values())
    ).style.format("{:.4f}", na_rep="—"))

signals = event_associations.loc[event_associations["p_value"].lt(0.05)].sort_values(["q_value", "p_value"])
display(Markdown("### Эффекты с p<0,05"))
show_tests(signals[["Бассейн", "Событие", "Фактор", "n₀", "n₁", "Без события", "С событием",
                    "Эффект", "Тест", "p_value", "q_value"]])

## Различия между бассейнами

In [ ]:
from itertools import combinations

basin_comparisons = []
for key, label in {"adverse_outcome": "Общий неблагоприятный исход", **event_labels}.items():
    for left, right in combinations(list(analysis_groups)[:3], 2):
        a, b = analysis_groups[left][key], analysis_groups[right][key]
        table = np.array([[int(a.eq(1).sum()), int(a.eq(0).sum())],
                          [int(b.eq(1).sum()), int(b.eq(0).sum())]])
        p = fisher_exact(table).pvalue if (table.sum(axis=0) > 0).all() else np.nan
        basin_comparisons.append({"Событие": label, "Сравнение": f"{left} / {right}",
                                  "p_value": p})
basin_comparisons = add_fdr(pd.DataFrame(basin_comparisons))
show_tests(basin_comparisons.sort_values(["Событие", "p_value"]))

## Итоги

Это описательный анализ зарегистрированных событий, а не оценка риска за фиксированный срок: даты обзвона и одинаковый период наблюдения не заданы. Анализируются строки исходной анкеты; повторные записи одного человека не гарантируют независимость наблюдений. Есть три пары одинаковых ФИО и дат операции, в том числе с противоречивыми событиями: их необходимо уточнить перед окончательными выводами. В описании одной реоперации указана будущая относительно 05.10.2026 дата 23.10.2026; событие закодировано по записи источника, но требует подтверждения.

ОШ>1 означает большую частоту исхода при наличии бинарного фактора, но не доказывает причинность. Отсутствие статистической значимости при редких осложнениях не доказывает отсутствие связи. Однофакторные сравнения выше не устанавливают независимые связи. Ниже добавлен отдельный многомерный анализ Фирта; сроки наблюдения и идентификация пациентов по-прежнему требуют уточнения.

In [ ]:
display(Markdown("### Связи с общим исходом после поправки FDR"))
robust_adverse = adverse_associations.loc[adverse_associations["q_value"].lt(0.05)]
if robust_adverse.empty:
    print("Для общего исхода связей с q<0,05 не выявлено.")
else:
    show_tests(robust_adverse[["Бассейн", "Фактор", "Эффект", "p_value", "q_value"]].sort_values("q_value"))
display(Markdown("### Связи с отдельными событиями после поправки FDR"))
robust_events = event_associations.loc[event_associations["q_value"].lt(0.05)]
if robust_events.empty:
    print("Для отдельных событий связей с q<0,05 не выявлено.")
else:
    show_tests(robust_events[["Бассейн", "Событие", "Фактор", "Эффект", "p_value", "q_value"]].sort_values("q_value"))

В текущем предварительном анализе после поправки FDR выделяется связь более высокого ЛПНП с общим неблагоприятным исходом в группе АБШ/аневризмы аорты. Для отдельных событий результатов с q<0,05 не выявлено. Это не означает отсутствия связей: часть событий редка, а размеры доступных групп зависят от пропусков. Вывод о ЛПНП также требует проверки после уточнения повторных записей и сроков наблюдения; причинное влияние по этим сравнениям не установлено.

## Регрессия Фирта по бассейнам

Три отдельные модели для `adverse_outcome`, а не только для смерти. Факторы: возраст на каждые 10 лет, женский пол, ЛПНП на 1 ммоль/л и сосудистые операции в анамнезе. Взаимодействия, отбор по p-value и синтетические данные не используются. Постоянные внутри бассейна признаки исключаются: их эффект нельзя оценить. В группе сонных артерий сосудистых операций в анамнезе нет у всех включённых пациентов, поэтому остаются три фактора; в двух других — четыре. Это ограниченные исследовательские модели, особенно при малом числе событий в группе сонных артерий. ЛПНП включён с учётом предыдущего анализа этих же данных, поэтому результаты не являются независимым подтверждением гипотезы.

Только для моделей формируется выборка с одной записью на сочетание ФИО и даты рождения внутри бассейна. При одинаковом известном исходе выбирается наиболее ранняя запись операции; пациенты с противоречивыми известными исходами исключаются. Затем исключаются строки с неизвестным исходом или пропусками в выбранных факторах. Это анализ полных наблюдений, потенциально подверженный смещению из-за пропусков. Исходные `df_all` и таблицы бассейнов не меняются. Идентификация по ФИО/дате рождения требует клинической проверки.

Используется пакет [firthmodels](https://pypi.org/project/firthmodels/) версии 0.8.2, установленный в текущем Python-окружении. Для другого окружения: `%pip install firthmodels==0.8.2`.


In [ ]:
from firthmodels import FirthLogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    roc_curve, precision_recall_curve, confusion_matrix,
)
from sklearn.calibration import calibration_curve

FIRTH_TARGET = "adverse_outcome"
FIRTH_FEATURES = ["age_years", "female", "ldl_mmol_l", "prior_vascular_surgery"]
FIRTH_THRESHOLD = 0.5
FIRTH_SEED = 42
firth_labels = ["Возраст, +10 лет", "Женский пол", "ЛПНП, +1 ммоль/л", "Операция на сосудах в анамнезе"]
firth_label_map = dict(zip(FIRTH_FEATURES, firth_labels))

firth_cohorts = {}
cohort_rows = []
for name in list(analysis_groups)[:3]:
    part = analysis_groups[name].copy()
    part["_patient"] = name_key(part["ФИО"]) + "|" + parse_dates(part["Год рождения"]).dt.strftime("%Y-%m-%d").fillna("")
    part["_operation"] = parse_dates(part["Дата операции"])
    conflicts = part.groupby("_patient")[FIRTH_TARGET].nunique().gt(1)
    conflicting_patients = conflicts.index[conflicts]
    selected = part.loc[~part["_patient"].isin(conflicting_patients)].dropna(subset=[FIRTH_TARGET])
    selected = selected.sort_values("_operation", kind="stable").drop_duplicates("_patient")
    complete = selected.dropna(subset=FIRTH_FEATURES).copy()
    firth_cohorts[name] = complete
    cohort_rows.append({"Бассейн": name, "Исходных строк": len(part),
                        "Пациентов с противоречивым исходом": len(conflicting_patients),
                        "Выбрано уникальных записей": len(selected),
                        "Исключено из-за пропусков факторов": len(selected)-len(complete),
                        "В модели": len(complete), "Событий": int(complete[FIRTH_TARGET].sum())})

firth_cohort_summary = pd.DataFrame(cohort_rows)
display(firth_cohort_summary)

## Оценки факторов

Для каждого бассейна модель оценивается на всей его модельной выборке. Это оценивание коэффициентов, не проверка качества прогнозирования. ОШ скорректировано на остальные включённые факторы. 95% ДИ — профильное штрафованное правдоподобие; p — штрафованный тест отношения правдоподобий. q — FDR для всех оцениваемых коэффициентов трёх моделей (11 в текущих данных). ДИ не скорректированы на множественные проверки. Метод: [Heinze и Schemper](https://doi.org/10.1002/sim.1047).


In [ ]:
def firth_design(part):
    design = part[FIRTH_FEATURES].astype(float).copy()
    design["age_years"] = design["age_years"] / 10
    return design.loc[:, design.nunique().gt(1)]

def fit_firth(design, target):
    model = FirthLogisticRegression(backend="numpy", max_iter=100, gtol=1e-6, xtol=1e-6)
    model.fit(design, target)
    if not model.converged_:
        raise RuntimeError("Регрессия Фирта не сошлась")
    return model

firth_models = {}
effect_rows = []
for name, part in firth_cohorts.items():
    design = firth_design(part)
    target = part[FIRTH_TARGET].to_numpy(dtype=int)
    model = fit_firth(design, target)
    included_features = design.columns.to_list()
    model.lrt(features=included_features)
    ci = model.conf_int(method="pl", features=included_features, max_iter=100)
    firth_models[name] = model
    for key in FIRTH_FEATURES:
        if key not in included_features:
            effect_rows.append({"Бассейн": name, "Фактор": firth_label_map[key], "ОШ": np.nan,
                                "ДИ 95% нижняя": np.nan, "ДИ 95% верхняя": np.nan,
                                "p_value": np.nan, "Статус": "Нет вариации фактора"})
            continue
        j = included_features.index(key)
        effect_rows.append({"Бассейн": name, "Фактор": firth_label_map[key],
                            "ОШ": np.exp(model.coef_[j]),
                            "ДИ 95% нижняя": np.exp(ci[j, 0]),
                            "ДИ 95% верхняя": np.exp(ci[j, 1]),
                            "p_value": model.lrt_pvalues_[j], "Статус": "Оценено"})

firth_effects = add_fdr(pd.DataFrame(effect_rows))
display(firth_effects.style.format({"ОШ": "{:.3f}", "ДИ 95% нижняя": "{:.3f}",
                                   "ДИ 95% верхняя": "{:.3f}", "p_value": "{:.4f}", "q_value": "{:.4f}"}, na_rep="—"))

fig, axes = plt.subplots(1, 3, figsize=(17, 4), sharey=True)
for ax, name in zip(axes, firth_cohorts):
    table = firth_effects.loc[firth_effects["Бассейн"].eq(name)]
    positions = np.arange(len(table))
    for pos, row in zip(positions, table.to_dict("records")):
        if pd.isna(row["ОШ"]):
            ax.text(0.5, pos, "Нет вариации", transform=ax.get_yaxis_transform(), ha="center", color="gray")
            continue
        ax.plot([row["ДИ 95% нижняя"], row["ДИ 95% верхняя"]], [pos, pos], color="#527A95")
        ax.scatter(row["ОШ"], pos, color="#B86B7D", zorder=3)
    ax.axvline(1, color="gray", linestyle="--")
    ax.set(xscale="log", xlabel="Скорректированное ОШ и 95% ДИ", title=name)
axes[0].set_yticks(np.arange(len(firth_labels)), firth_labels)
axes[0].invert_yaxis()
plt.tight_layout()
plt.show()

## Качество вне обучения

Пятикратная стратифицированная кросс-валидация отдельно внутри каждого бассейна. Каждый пациент получает прогноз модели, обученной без его записи; итоговые прогнозы — OOF. Набор факторов фиксирован, параметры и порог на тестовых частях не подбираются. Константный ориентир предсказывает частоту события только в обучающей части каждого разбиения.

ROC-AUC характеризует ранжирование; AP — average precision по precision–recall, не площадь методом трапеций. Brier — средний квадрат ошибки вероятности, меньше лучше. Для ROC-AUC дан приближённый 95% интервал по 1000 бутстрэп-выборкам готовых OOF-прогнозов: он не учитывает полностью неопределённость повторного обучения и зависимость моделей между фолдами. Это внутренняя исследовательская оценка, не внешняя валидация; предварительный выбор ЛПНП по этим же данным может придавать ей оптимизм.


In [ ]:
def auc_interval(target, probability, n_boot=1000):
    rng = np.random.default_rng(FIRTH_SEED)
    estimates = []
    for _ in range(n_boot):
        indices = rng.integers(0, len(target), size=len(target))
        if np.unique(target[indices]).size == 2:
            estimates.append(roc_auc_score(target[indices], probability[indices]))
    return np.quantile(estimates, [0.025, 0.975])

firth_oof = {}
quality_rows = []
threshold_rows = []
for name, part in firth_cohorts.items():
    design = firth_design(part)
    target = part[FIRTH_TARGET].to_numpy(dtype=int)
    probability = np.full(len(part), np.nan)
    baseline_probability = np.full(len(part), np.nan)
    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=FIRTH_SEED)
    for train, test in folds.split(design, target):
        model = fit_firth(design.iloc[train], target[train])
        probability[test] = model.predict_proba(design.iloc[test])[:, 1]
        baseline_probability[test] = target[train].mean()
    if not np.isfinite(probability).all():
        raise RuntimeError("Не для всех пациентов получены OOF-прогнозы")
    low, high = auc_interval(target, probability)
    firth_oof[name] = {"target": target, "probability": probability,
                       "baseline_probability": baseline_probability}
    quality_rows.append({"Бассейн": name, "n": len(part), "Событий": int(target.sum()),
                         "ROC-AUC": roc_auc_score(target, probability), "AUC ДИ нижняя": low,
                         "AUC ДИ верхняя": high, "AP": average_precision_score(target, probability),
                         "Частота события": target.mean(), "Brier": brier_score_loss(target, probability),
                         "Brier константы": brier_score_loss(target, baseline_probability)})
    tn, fp, fn, tp = confusion_matrix(target, probability >= FIRTH_THRESHOLD, labels=[0, 1]).ravel()
    threshold_rows.append({"Бассейн": name, "Порог": FIRTH_THRESHOLD,
                           "Recall события": tp/(tp+fn), "Специфичность": tn/(tn+fp),
                           "Precision": tp/(tp+fp) if tp+fp else np.nan,
                           "F1": 2*tp/(2*tp+fp+fn), "TP": tp, "FN": fn, "FP": fp, "TN": tn})

firth_quality = pd.DataFrame(quality_rows)
firth_threshold_metrics = pd.DataFrame(threshold_rows)
quality_display = firth_quality.drop(columns=["AUC ДИ нижняя", "AUC ДИ верхняя"]).copy()
quality_display["ROC-AUC"] = [f"{r['ROC-AUC']:.3f} [{r['AUC ДИ нижняя']:.3f}; {r['AUC ДИ верхняя']:.3f}]"
                              for r in quality_rows]
display(quality_display.style.format({"AP": "{:.3f}", "Частота события": "{:.3f}",
                                      "Brier": "{:.3f}", "Brier константы": "{:.3f}"}))
display(firth_threshold_metrics.drop(columns=["TP", "FN", "FP", "TN"]).style.format(
    {"Порог": "{:.2f}", "Recall события": "{:.3f}", "Специфичность": "{:.3f}",
     "Precision": "{:.3f}", "F1": "{:.3f}"}, na_rep="—"))

## ROC, PR и калибровка

Все кривые используют OOF-прогнозы. В PR пунктир обозначает частоту события в соответствующей выборке. Калибровка сопоставляет средний прогноз и наблюдаемую частоту в пяти группах по квантилям вероятности; в малой выборке точки нестабильны. Хорошее ранжирование само по себе не гарантирует точные вероятности. Обычная регрессия Фирта может смещать редкие прогнозируемые вероятности к 0,5; здесь они не корректировались дополнительным методом. [Puhr и соавторы, 2017](https://pubmed.ncbi.nlm.nih.gov/28295456/).


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
model_colors = dict(zip(firth_cohorts, ["#527A95", "#B86B7D", "#6F9A87"]))
for name, result in firth_oof.items():
    target, probability = result["target"], result["probability"]
    color = model_colors[name]
    fpr, tpr, _ = roc_curve(target, probability)
    axes[0].plot(fpr, tpr, color=color, label=f"{name}: AUC={roc_auc_score(target, probability):.3f}")
    precision, recall, _ = precision_recall_curve(target, probability)
    axes[1].step(recall, precision, where="post", color=color,
                 label=f"{name}: AP={average_precision_score(target, probability):.3f}")
    axes[1].axhline(target.mean(), color=color, linestyle="--", alpha=0.6)
    observed, predicted = calibration_curve(target, probability, n_bins=5, strategy="quantile")
    axes[2].plot(predicted, observed, "o-", color=color, label=name)

axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[2].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set(title="ROC: прогнозы вне обучения", xlabel="1 − специфичность", ylabel="Чувствительность")
axes[1].set(title="Precision–Recall", xlabel="Recall события", ylabel="Precision")
axes[2].set(title="Калибровка", xlabel="Средняя прогнозируемая вероятность", ylabel="Наблюдаемая частота")
for ax in axes:
    ax.set(xlim=(0, 1), ylim=(0, 1))
    ax.legend(fontsize=7, loc="best")
plt.tight_layout()
plt.show()

## Ошибки при пороге 0,5

Положительный класс — общий неблагоприятный исход. Recall — доля выявленных событий; специфичность — доля правильно распознанных отсутствий событий; precision — доля настоящих событий среди положительных прогнозов. Если положительных прогнозов нет, precision не определена и показана как «—». Порог 0,5 фиксирован для прозрачного сравнения, не является клинически выбранным. Низкий recall при нём не равнозначен плохому ранжированию: это отдельно оценивает ROC-AUC.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, result) in zip(axes, firth_oof.items()):
    matrix = confusion_matrix(result["target"], result["probability"] >= FIRTH_THRESHOLD, labels=[0, 1])
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Нет события", "Есть событие"], yticklabels=["Нет события", "Есть событие"])
    ax.set(title=f"{name}\nПорог {FIRTH_THRESHOLD:.2f}", xlabel="Прогноз", ylabel="Фактический исход")
plt.tight_layout()
plt.show()

## Выводы по моделям

Статистическая связь фактора и способность прогнозировать исход — разные результаты. ОШ ниже относятся к небольшим моделям на доступных полных наблюдениях. Проверка качества выполнена отдельно через OOF. Нет оценки риска за одинаковый срок, внешней валидации и оснований для применения этих прогнозов в клинических решениях.


In [ ]:
for row in quality_rows:
    name = row["Бассейн"]
    threshold = firth_threshold_metrics.loc[firth_threshold_metrics["Бассейн"].eq(name)].iloc[0]
    baseline_comparison = "ниже константного ориентира" if row["Brier"] < row["Brier константы"] else "не лучше константного ориентира"
    display(Markdown(
        f"**{name}**: n={row['n']}, событий {row['Событий']}. "
        f"OOF ROC-AUC={row['ROC-AUC']:.3f} [{row['AUC ДИ нижняя']:.3f}; {row['AUC ДИ верхняя']:.3f}], "
        f"AP={row['AP']:.3f} при частоте события {row['Частота события']:.3f}. "
        f"Brier={row['Brier']:.3f}, {baseline_comparison} ({row['Brier константы']:.3f}). "
        f"При пороге {FIRTH_THRESHOLD:.2f} recall={threshold['Recall события']:.3f}, "
        f"специфичность={threshold['Специфичность']:.3f}."
    ))

model_signals = firth_effects.loc[firth_effects["q_value"].lt(0.05)]
if model_signals.empty:
    print("Скорректированных связей факторов с q<0,05 в этих моделях не выявлено.")
else:
    display(Markdown("### Скорректированные связи с q<0,05"))
    display(model_signals.style.format({"ОШ": "{:.3f}", "ДИ 95% нижняя": "{:.3f}",
                                       "ДИ 95% верхняя": "{:.3f}", "p_value": "{:.4f}", "q_value": "{:.4f}"}))